# 04 — Jira Effort Estimation: Data Preparation

Continúa desde [`01_eda.ipynb`](./01_eda.ipynb). Este notebook:

1. Combina título + descripción en un solo texto y genera **embeddings** con `sentence-transformers`.
2. Aplica `log1p` al target (`storypoint`), ya justificado en el EDA por el sesgo extremo de la distribución.
3. Hace un split **cronológico por proyecto** (no aleatorio), ordenando explícitamente por número de issue key dentro de cada proyecto — se verifica primero cuánto se aparta el orden de fila del CSV del orden real de creación (no es perfecto, ver sección 3), así que no se asume, se corrige antes de dividir train/test.
4. Deja el dataset preparado para comparar, en `03_modeling.ipynb`, un modelo *pooled* (con `project` como feature) contra modelos *por proyecto* — la alternativa que el EDA dejó planteada frente a la heterogeneidad de escalas entre proyectos.

In [1]:
import glob
import os

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_columns', None)
DATA_DIR = '../data'

## 1. Cargar y combinar los 16 proyectos

In [2]:
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))

dfs = []
for f in files:
    project_name = os.path.splitext(os.path.basename(f))[0]
    d = pd.read_csv(f)
    d['project'] = project_name
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
print(df.shape)

(23313, 5)


## 2. Texto combinado y target transformado

In [3]:
df['description'] = df['description'].fillna('')
df['combined_text'] = (df['title'] + '. ' + df['description']).str.strip()
df['log_storypoint'] = np.log1p(df['storypoint'])

df[['issuekey', 'project', 'combined_text', 'storypoint', 'log_storypoint']].head(3)

,issuekey,project,combined_text,storypoint,log_storypoint
0,TISTUD-6,appceleratorstudio,Add CA against object literals in function inv...,1,0.693147
1,TISTUD-9,appceleratorstudio,Update branding for Appcelerator plugin to App...,1,0.693147
2,TISTUD-11,appceleratorstudio,Create new JSON schema for SDK team. {html}<di...,1,0.693147


## 3. Split cronológico por proyecto

Se verifica primero si el orden de las filas dentro de cada proyecto coincide con el orden de creación del issue (número de issue key creciente) — si no fuera así, un split "cronológico" por posición de fila sería inválido.

In [4]:
df['issue_num'] = df['issuekey'].str.extract(r'-(\d+)$').astype(int)

def pct_out_of_order(s):
    return (s.diff() < 0).mean()

violation_rate = df.groupby('project')['issue_num'].apply(pct_out_of_order).sort_values(ascending=False)
print('% de filas fuera de orden por proyecto (issue_num decrece respecto a la fila anterior):')
print((violation_rate * 100).round(2))

% de filas fuera de orden por proyecto (issue_num decrece respecto a la fila anterior):
project
titanium              13.77
appceleratorstudio     3.77
mule                   2.47
mulestudio             2.05
talenddataquality      2.03
aptanastudio           1.57
clover                 1.04
talendesb              1.04
jirasoftware           0.85
moodle                 0.09
springxd               0.06
datamanagement         0.02
mesos                  0.00
duracloud              0.00
bamboo                 0.00
usergrid               0.00
Name: issue_num, dtype: float64


**Resultado real, no el esperado:** el orden de fila casi siempre coincide con el orden de creación, pero no es perfecto — la mayoría de los proyectos tiene entre 0% y 4% de filas puntualmente fuera de orden (probablemente subtareas o issues reabiertos/renumerados), y `titanium` llega a 13.8%. No es aceptable asumir el orden de fila a ciegas. **Corrección:** en vez de usar la posición de fila, se ordena explícitamente por `issue_num` dentro de cada proyecto antes de cortar train/test — así el split es cronológicamente correcto por construcción, no por una suposición sobre el CSV de origen.

In [5]:
TEST_FRAC = 0.2

train_parts, test_parts = [], []
for project, group in df.groupby('project', sort=False):
    group = group.sort_values('issue_num')  # orden cronológico garantizado, no asumido
    cutoff = int(len(group) * (1 - TEST_FRAC))
    train_parts.append(group.iloc[:cutoff])
    test_parts.append(group.iloc[cutoff:])

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print()
print('Filas de train/test por proyecto (primeras 5):')
print(pd.DataFrame({'train': train_df['project'].value_counts(), 'test': test_df['project'].value_counts()}).head())

Train: 18642 | Test: 4671



Filas de train/test por proyecto (primeras 5):
                    train  test
project                        
datamanagement       3733   934
springxd             2820   706
appceleratorstudio   2335   584
titanium             1800   451
mesos                1344   336


El split se hace **dentro de cada proyecto** (no cortando por una fecha global) — cada proyecto tiene su propio calendario de desarrollo, así que lo relevante es "los issues más recientes de *ese* proyecto", no una fecha absoluta compartida entre los 16.

## 4. Embeddings de texto

Se usa `sentence-transformers/all-MiniLM-L6-v2` (384 dimensiones) — modelo liviano, buen equilibrio entre calidad y velocidad para ~23K textos cortos de tickets de software.

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

train_embeddings = model.encode(train_df['combined_text'].tolist(), batch_size=64, show_progress_bar=True)
test_embeddings = model.encode(test_df['combined_text'].tolist(), batch_size=64, show_progress_bar=True)

print('train_embeddings:', train_embeddings.shape)
print('test_embeddings:', test_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/292 [00:00<?, ?it/s]

Batches:   0%|          | 0/73 [00:00<?, ?it/s]

train_embeddings: (18642, 384)
test_embeddings: (4671, 384)


## 5. Guardado de datasets procesados

Los embeddings se guardan aparte (`.npy`, más eficiente que texto para una matriz densa de floats) de los metadatos (`.csv`, con `project`, `storypoint` y `log_storypoint` — necesarios para el modelo pooled-con-feature-de-proyecto y para la comparación por proyecto). Ninguno se versiona en git.

In [7]:
train_df[['issuekey', 'project', 'storypoint', 'log_storypoint']].to_csv(f'{DATA_DIR}/processed_train.csv', index=False)
test_df[['issuekey', 'project', 'storypoint', 'log_storypoint']].to_csv(f'{DATA_DIR}/processed_test.csv', index=False)

np.save(f'{DATA_DIR}/train_embeddings.npy', train_embeddings)
np.save(f'{DATA_DIR}/test_embeddings.npy', test_embeddings)

print('Guardado: processed_train.csv, processed_test.csv, train_embeddings.npy, test_embeddings.npy')

Guardado: processed_train.csv, processed_test.csv, train_embeddings.npy, test_embeddings.npy


## 6. Conclusiones de la fase de Data Preparation

1. **Texto combinado** (`title` + `description`, con nulos de descripción como vacío) vectorizado con `all-MiniLM-L6-v2` (384-dim).
2. **Target log-transformado** (`log_storypoint`) para atenuar el sesgo extremo detectado en el EDA — se evalúa en escala original (`storypoint`) revirtiendo la transformación, no en escala log, para que el error sea interpretable en "puntos".
3. **Split cronológico dentro de cada proyecto, corregido en base a un hallazgo real:** el orden de fila del CSV no coincide perfectamente con el orden de creación (hasta 13.8% de filas fuera de orden en `titanium`) — se ordena explícitamente por número de issue key antes de cortar train/test, en vez de asumir el orden del archivo. 80% más antiguo a train, 20% más reciente a test, por proyecto.
4. **`project` se conserva como columna** en los metadatos procesados, para poder comparar en `03_modeling.ipynb` un modelo pooled con `project` como feature categórica contra el enfoque de modelos separados por proyecto que el propio paper de origen de este dataset utiliza — la respuesta directa al hallazgo de heterogeneidad de escalas del EDA.

**Siguiente paso:** `03_modeling.ipynb` — regresión sobre los embeddings, comparando ambos enfoques.